# 03 — Feature engineering
Use the existing, reviewed topology implementation. The 22-column dataset represents each player's position at the end of setup. Pips are counts of equally likely dice outcomes (out of 36), not actual observed production. Expansion options assume a frozen initial board.

In [1]:
from pathlib import Path
from importlib import import_module
import pandas as pd

OUT = Path("processed_data_5000")
fe = import_module("03_feature_engineering")
placements = pd.read_csv(OUT / "initial_placements.csv")
tiles = pd.read_csv(OUT / "map_tiles.csv")
features = fe.make_features(placements, tiles)
assert len(features) == len(placements)
assert not features.isna().any().any()
assert not features.duplicated(["Game_ID","Player"]).any()
assert features.Total_Production.eq(features.Settlement_1_Score + features.Settlement_2_Score).all()
production = [r.title()+"_Production" for r in fe.RESOURCES]
assert features.Total_Production.eq(features[production].sum(axis=1)).all()
assert features.Initial_Total.eq(features[["Initial_"+r.title() for r in fe.RESOURCES]].sum(axis=1)).all()
display(features.head())
print(features.shape)

,Game_ID,Player,Placement_Order,Settlement_1_Score,Settlement_2_Score,Wood_Production,Brick_Production,Sheep_Production,Wheat_Production,Ore_Production,...,Resource_Diversity,Number_Diversity,Any_Production_Probability,Initial_Wood,Initial_Brick,Initial_Sheep,Initial_Wheat,Initial_Ore,Initial_Total,Expansion_Options
0,000a8611-8efa-43ef-aa17-2b695845f514,ORANGE,1,6,3,0,0,3,5,1,...,3,3,0.250000,0,0,1,0,0,1,2
1,000a8611-8efa-43ef-aa17-2b695845f514,RED,2,5,9,0,0,4,5,5,...,3,4,0.388889,0,0,1,2,0,3,2
2,000a8611-8efa-43ef-aa17-2b695845f514,BLUE,3,9,10,4,1,6,5,3,...,5,6,0.527778,0,0,1,1,1,3,1
3,000a8611-8efa-43ef-aa17-2b695845f514,WHITE,4,5,7,0,5,0,2,5,...,3,2,0.194444,0,1,0,1,0,2,1
4,00248db7-8590-49d2-a14c-aa3e61ff5128,RED,1,0,7,0,2,5,0,0,...,2,3,0.194444,0,1,2,0,0,3,3


(20000, 22)


In [2]:
# Rebuild the feature CSV only if needed. This is a local file operation.
features.to_csv(OUT / "initial_features.csv", index=False)
display(features.select_dtypes("number").describe().T)

,count,mean,std,min,25%,50%,75%,max
Placement_Order,20000.0,2.500000,1.118062,1.000000,1.750000,2.500000,3.250000,4.000000
Settlement_1_Score,20000.0,6.414600,3.136367,0.000000,4.000000,7.000000,9.000000,13.000000
Settlement_2_Score,20000.0,6.312450,3.170083,0.000000,4.000000,6.000000,9.000000,13.000000
Wood_Production,20000.0,2.832650,2.892968,0.000000,0.000000,2.000000,5.000000,17.000000
Brick_Production,20000.0,2.110200,2.530055,0.000000,0.000000,1.000000,4.000000,17.000000
Sheep_Production,20000.0,2.808900,2.880676,0.000000,0.000000,2.000000,5.000000,17.000000
Wheat_Production,20000.0,2.873800,2.918715,0.000000,0.000000,2.000000,5.000000,16.000000
Ore_Production,20000.0,2.101500,2.555372,0.000000,0.000000,1.000000,4.000000,14.000000
Total_Production,20000.0,12.727050,4.317103,1.000000,10.000000,13.000000,16.000000,25.000000
Expected_Cards_Per_Roll,20000.0,0.353529,0.119920,0.027778,0.277778,0.361111,0.444444,0.694444


Excluded: final board, dice outcomes, winner, third settlement, player IDs as predictors. Port types are not part of the core feature CSV; they are a possible extension. Aggregate columns are retained for interpretation but redundant columns will be excluded from modeling.